# Modelimi në Fizikë — Provim i pjesshëm I

**Koha në dispozicion:** 40 minuta  
**Totali:** 40 pikë  
**Emri dhe mbiemri:** ____________________________  
**Data:** ____________________________

Ky provim vlerëson aftësinë për të kaluar nga formulimi fizik te modeli numerik, për të implementuar integratorë bazë të ODE-ve dhe për të interpretuar rezultatet numerike në mënyrë kritike.


## Udhëzime

- Plotësoni vetëm pjesët e shënuara me `TODO` ose `...`.
- Mos ndryshoni emrat e funksioneve të dhëna, sepse ato mund të përdoren për vlerësim automatik.
- Çdo figurë duhet të ketë titull, etiketa boshtesh dhe legjendë aty ku kërkohet.
- Përgjigjet teorike duhet të jenë të shkurtra, por të argumentuara fizikisht.
- Lejohet përdorimi i `numpy` dhe `matplotlib`. Nuk kërkohet përdorimi i `scipy`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True


---
## Ushtrimi 1 — Relaksimi eksponencial dhe kalibrimi i një parametri (10 pikë)

Konsideroni modelin

$$
\frac{dx}{dt}=-kx, \qquad x(0)=x_0 .
$$

Do të përdorni metodën Euler për të simuluar modelin dhe më pas do të gjeni vlerën e parametrit \(k\) që përshtatet më mirë me disa të dhëna sintetike.

**Pikëzimi:**

- 3 pikë: implementimi korrekt i hapit Euler;
- 3 pikë: ndërtimi korrekt i funksionit të gabimit mesatar kuadratik;
- 2 pikë: kërkimi në rrjetë për \(k\);
- 2 pikë: interpretim i shkurtër fizik/numerik.


In [ ]:
def simulate_relaxation_euler(k, x0=1.0, t_end=5.0, dt=0.01):
    # Simulon dx/dt = -k x me metodën Euler.
    t = np.arange(0.0, t_end + dt, dt)
    x = np.zeros_like(t)
    x[0] = x0

    for n in range(len(t) - 1):
        # TODO 1.1: shkruani derivatin dx/dt
        dxdt = ...

        # TODO 1.2: përditësoni x[n+1] sipas metodës Euler
        x[n + 1] = ...

    return t, x


def mse_for_k(k, t_data, x_data, x0=1.0, dt=0.01):
    # Kthen gabimin mesatar kuadratik midis modelit dhe të dhënave.
    t_model, x_model = simulate_relaxation_euler(
        k=k,
        x0=x0,
        t_end=float(t_data[-1]),
        dt=dt
    )

    # TODO 1.3: llogaritni MSE midis x_model dhe x_data
    mse = ...
    return float(mse)


In [ ]:
# Të dhëna sintetike për provim: mos i ndryshoni.
rng = np.random.default_rng(42)
t_data = np.arange(0.0, 5.0 + 0.01, 0.01)
k_true_hidden = 1.35
x_clean = np.exp(-k_true_hidden * t_data)
x_data = x_clean + 0.015 * rng.standard_normal(len(t_data))

# TODO 1.4: krijoni një rrjetë vlerash për k në intervalin [0.5, 2.2]
k_grid = ...

# TODO 1.5: llogaritni gabimin për çdo k dhe gjeni k_best
losses = ...
k_best = ...

print("k_best =", k_best)
print("MSE minimal =", np.min(losses))

plt.figure()
plt.plot(t_data, x_data, ".", markersize=2, label="të dhëna sintetike")

# TODO 1.6: simuloni modelin me k_best dhe vizatoni kurbën e përshtatur
# t_fit, x_fit = ...
# plt.plot(...)

plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Relaksimi eksponencial dhe përshtatja e parametrit k")
plt.legend()
plt.show()


**Përgjigje e shkurtër për Ushtrimin 1:**  
Shpjegoni me 2–3 fjali pse gabimi minimal jep një vlerësim të arsyeshëm të parametrit \(k\), dhe përmendni një kufizim të kësaj procedure.

_Përgjigjja juaj:_


---
## Ushtrimi 2 — Oscilatori harmonik, RK4 dhe ruajtja e energjisë (14 pikë)

Oscilatori harmonik pa fërkim jepet nga

$$
\ddot{x}+\omega^2x=0, \qquad \dot{x}=v, \qquad \dot{v}=-\omega^2x.
$$

Energjia teorike është

$$
E=\frac{1}{2}v^2+\frac{1}{2}\omega^2x^2.
$$

**Detyra:** Plotësoni funksionin e anës së djathtë, hapin RK4 dhe funksionin e energjisë. Pastaj krahasoni evolucionin e energjisë me metodën Euler dhe RK4.

**Pikëzimi:**

- 3 pikë: formulimi i saktë i sistemit të rendit të parë;
- 5 pikë: implementimi i saktë i RK4;
- 3 pikë: llogaritja dhe paraqitja e gabimit relativ të energjisë;
- 3 pikë: interpretimi fizik i drift-it të energjisë.


In [ ]:
def rhs_oscillator(state, omega):
    # Kthen [dx/dt, dv/dt] për oshilatorin harmonik.
    x, v = state

    # TODO 2.1: plotësoni sistemin e rendit të parë
    dxdt = ...
    dvdt = ...
    return np.array([dxdt, dvdt], dtype=float)


def euler_step(state, dt, omega):
    return state + dt * rhs_oscillator(state, omega)


def rk4_step(state, dt, omega):
    # Një hap i metodës Runge-Kutta të rendit të katërt.
    # TODO 2.2: plotësoni k1, k2, k3, k4 dhe formulën finale
    k1 = ...
    k2 = ...
    k3 = ...
    k4 = ...
    next_state = ...
    return next_state


def integrate_oscillator(step_function, state0, omega=2.0, t_end=30.0, dt=0.05):
    t = np.arange(0.0, t_end + dt, dt)
    y = np.zeros((len(t), 2), dtype=float)
    y[0] = np.array(state0, dtype=float)

    for n in range(len(t) - 1):
        y[n + 1] = step_function(y[n], dt, omega)

    return t, y


def energy_oscillator(x, v, omega):
    # TODO 2.3: plotësoni energjinë totale
    return ...


In [ ]:
omega = 2.0
state0 = np.array([1.0, 0.0])
t_end = 30.0
dt = 0.05

t_eu, y_eu = integrate_oscillator(euler_step, state0, omega, t_end, dt)
t_rk, y_rk = integrate_oscillator(rk4_step, state0, omega, t_end, dt)

E_eu = energy_oscillator(y_eu[:, 0], y_eu[:, 1], omega)
E_rk = energy_oscillator(y_rk[:, 0], y_rk[:, 1], omega)
E0 = energy_oscillator(state0[0], state0[1], omega)

plt.figure()
plt.plot(t_eu, y_eu[:, 0], label="Euler")
plt.plot(t_rk, y_rk[:, 0], label="RK4")
plt.xlabel("t")
plt.ylabel("x(t)")
plt.title("Trajektorja e oshilatorit harmonik")
plt.legend()
plt.show()

plt.figure()
plt.plot(t_eu, (E_eu - E0) / E0, label="Euler")
plt.plot(t_rk, (E_rk - E0) / E0, label="RK4")
plt.xlabel("t")
plt.ylabel(r"$(E(t)-E_0)/E_0$")
plt.title("Gabimi relativ i energjisë")
plt.legend()
plt.show()


**Përgjigje e shkurtër për Ushtrimin 2:**  
Cila metodë sillet më mirë për energjinë në këtë provë dhe pse? A mjafton vetëm saktësia lokale për të garantuar sjellje fizike afatgjatë?

_Përgjigjja juaj:_


---
## Ushtrimi 3 — Lavjerrësi jolinear dhe kufiri i këndeve të vogla (10 pikë)

Lavjerrësi matematik jolinear përshkruhet nga

$$
\ddot{\theta}=-\frac{g}{L}\sin\theta.
$$

Për kënde të vogla, periudha afrohet nga

$$
T_0 = 2\pi \sqrt{\frac{L}{g}}.
$$

**Detyra:** Plotësoni modelin, simuloni dy kushte fillestare dhe vlerësoni periudhën nga kalimet \(\theta=0\).

**Pikëzimi:**

- 3 pikë: formulimi i saktë i sistemit jolinear;
- 3 pikë: simulimi për dy amplitude fillestare;
- 2 pikë: vlerësimi numerik i periudhës;
- 2 pikë: krahasimi me formulën e këndeve të vogla.


In [ ]:
def rhs_pendulum(state, g=9.81, L=1.0):
    theta, omega_theta = state

    # TODO 3.1: plotësoni dtheta/dt dhe domega/dt
    dtheta_dt = ...
    domega_dt = ...
    return np.array([dtheta_dt, domega_dt], dtype=float)


def rk4_step_general(rhs, state, dt, *args, **kwargs):
    k1 = rhs(state, *args, **kwargs)
    k2 = rhs(state + 0.5 * dt * k1, *args, **kwargs)
    k3 = rhs(state + 0.5 * dt * k2, *args, **kwargs)
    k4 = rhs(state + dt * k3, *args, **kwargs)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


def integrate_pendulum(theta0, omega0=0.0, g=9.81, L=1.0, t_end=20.0, dt=0.01):
    t = np.arange(0.0, t_end + dt, dt)
    y = np.zeros((len(t), 2), dtype=float)
    y[0] = [theta0, omega0]

    for n in range(len(t) - 1):
        y[n + 1] = rk4_step_general(rhs_pendulum, y[n], dt, g=g, L=L)

    return t, y


def estimate_period_from_zero_crossings(t, theta):
    # Vlerëson periudhën nga kalimet me pjerrësi pozitive për theta=0.
    crossings = []
    for n in range(len(theta) - 1):
        # TODO 3.2: gjeni kalimet nga vlera negative në pozitive
        if ...:
            # interpolim linear për kohën e kalimit
            tc = t[n] - theta[n] * (t[n+1] - t[n]) / (theta[n+1] - theta[n])
            crossings.append(tc)

    crossings = np.array(crossings)
    if len(crossings) < 2:
        return np.nan

    # TODO 3.3: periudha është diferenca mes kalimeve të njëpasnjëshme me të njëjtën fazë
    return ...


In [ ]:
angles = [0.10, 1.20]  # radianë
results = {}

plt.figure()
for theta0 in angles:
    t, y = integrate_pendulum(theta0=theta0, t_end=20.0, dt=0.01)
    T_est = estimate_period_from_zero_crossings(t, y[:, 0])
    results[theta0] = T_est
    plt.plot(t, y[:, 0], label=fr"$\theta_0={theta0}$ rad, T≈{T_est:.3f}")

T_small = 2 * np.pi * np.sqrt(1.0 / 9.81)
print("T_small_angle =", T_small)
print("T_estimates =", results)

plt.xlabel("t [s]")
plt.ylabel(r"$\theta(t)$ [rad]")
plt.title("Lavjerrësi jolinear për dy amplituda fillestare")
plt.legend()
plt.show()


**Përgjigje e shkurtër për Ushtrimin 3:**  
Pse periudha e lavjerrësit me amplitudë të madhe ndryshon nga \(T_0\)? Cili është kuptimi fizik i përafrimit \(\sin\theta\approx\theta\)?

_Përgjigjja juaj:_


---
## Ushtrimi 4 — Arsyetim konceptual mbi modelimin numerik (6 pikë)

Përgjigjuni shkurt, me fjali të plota.

1. Cili është dallimi midis **simulimit**, **optimizimit** dhe **problemit invers** në modelimin në fizikë? (3 pikë)
2. Përse kodi numerik duhet strukturuar në funksione të veçanta dhe jo vetëm si qeliza të gjata të njëpasnjëshme? (3 pikë)

_Përgjigjja juaj:_
